# Final Bicubic Evaluation — All Datasets and Scales

Run every cell from top to bottom in a fresh Google Colab session. This evaluates Set5, Set14, BSD100, and Urban100 at x2, x3, and x4 using the approved final protocol. Outputs are stored in a new timestamped Google Drive folder.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/divinesta/SuperResolution-Comparative-Analysis.git"
REPO_ROOT = Path("/content/SuperResolution-Comparative-Analysis")

if REPO_ROOT.exists():
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
print(f"Repository ready: {REPO_ROOT}")

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")],
    check=True,
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import skimage
from PIL import __version__ as pillow_version
from app.evaluation.metrics import calculate_quality_metrics

print("Dependencies installed and metric imports verified.")
print(f"NumPy: {np.__version__}")
print(f"Pillow: {pillow_version}")
print(f"scikit-image: {skimage.__version__}")

In [ ]:
from datetime import UTC, datetime

DATA_ROOT = Path("/content/drive/MyDrive/FYP_SR_Data")
RUN_ID = datetime.now(UTC).strftime("%Y%m%d_%H%M%S_utc")
RUN_ROOT = DATA_ROOT / "results" / "final_bicubic" / "full" / RUN_ID
METRICS_ROOT = RUN_ROOT / "metrics"
DATASETS = ("Set5", "Set14", "BSD100", "Urban100")
SCALES = (2, 3, 4)

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset root not found: {DATA_ROOT}")

print(f"Data root: {DATA_ROOT}")
print(f"Full run will save to: {RUN_ROOT}")

In [ ]:
from app.evaluation.data_validation import validate_prepared_dataset

validations = {}
for dataset in DATASETS:
    for scale in SCALES:
        validation = validate_prepared_dataset(dataset, scale, DATA_ROOT)
        validations[(dataset, scale)] = validation
        print(
            f"VALID: {dataset} x{scale} has "
            f"{validation.image_count} complete HR/LR pairs."
        )

print("All 12 dataset/scale combinations passed validation.")

In [ ]:
from app.evaluation.bicubic import (
    BicubicEvaluationConfig,
    evaluate_bicubic_dataset,
    write_results_csv,
)

all_records = []
for dataset in DATASETS:
    for scale in SCALES:
        validation = validations[(dataset, scale)]
        print(f"Running {dataset} x{scale}...")
        config = BicubicEvaluationConfig(
            dataset=dataset,
            scale=scale,
            warmup_runs=3,
            timed_runs=10,
        )
        records = evaluate_bicubic_dataset(
            validation.hr_directory,
            validation.lr_directory,
            config,
        )
        combination_csv = METRICS_ROOT / f"{dataset}_x{scale}_bicubic_final.csv"
        write_results_csv(records, combination_csv)
        all_records.extend(records)
        print(f"Completed {dataset} x{scale}: {len(records)} images.")

print(f"Full bicubic evaluation completed: {len(all_records)} image evaluations.")

In [ ]:
from app.evaluation.reporting import summarize_results

combined_csv = METRICS_ROOT / "bicubic_all_images_final.csv"
summary_csv = METRICS_ROOT / "bicubic_summary_final.csv"
summary_records = summarize_results(all_records)
write_results_csv(all_records, combined_csv)
write_results_csv(summary_records, summary_csv)

header = (
    f"{'Dataset':<10} {'Scale':<6} {'Images':>6} "
    f"{'Y PSNR':>10} {'Y SSIM':>10} {'RGB PSNR':>10} "
    f"{'RGB SSIM':>10} {'Latency':>12}"
)
print(header)
print("-" * len(header))
for row in summary_records:
    print(
        f"{row['dataset']:<10} {row['scale']:<6} {row['image_count']:>6} "
        f"{row['psnr_y']:>10.4f} {row['ssim_y']:>10.4f} "
        f"{row['psnr_rgb']:>10.4f} {row['ssim_rgb']:>10.4f} "
        f"{row['latency_mean_ms']:>10.4f} ms"
    )

print(f"Combined results: {combined_csv}")
print(f"Summary results: {summary_csv}")

## After the run

Send the printed 12-row summary and the generated `bicubic_summary_final.csv` file for review. Do not overwrite or delete the earlier preliminary or pilot results.